In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

In [17]:

for dirpath, dirnames, filenames in os.walk("/mnt/e/Kitti/training/image_02"):
    # print(dirpath, len(filenames))
    if len(filenames) > 0:
        print(dirpath, filenames[0])
    # if filenames:
    #     print(f"  Files: {', '.join(filenames)}")
    # print("-" * 20) # Separator for clarity

/mnt/e/Kitti/training/image_02/0000 000027.png
/mnt/e/Kitti/training/image_02/0001 000000.png
/mnt/e/Kitti/training/image_02/0002 000000.png
/mnt/e/Kitti/training/image_02/0003 000034.png
/mnt/e/Kitti/training/image_02/0004 000000.png
/mnt/e/Kitti/training/image_02/0005 000000.png
/mnt/e/Kitti/training/image_02/0006 000000.png
/mnt/e/Kitti/training/image_02/0007 000000.png
/mnt/e/Kitti/training/image_02/0008 000000.png
/mnt/e/Kitti/training/image_02/0009 000000.png
/mnt/e/Kitti/training/image_02/0010 000000.png
/mnt/e/Kitti/training/image_02/0011 000000.png
/mnt/e/Kitti/training/image_02/0012 000022.png
/mnt/e/Kitti/training/image_02/0013 000000.png
/mnt/e/Kitti/training/image_02/0014 000032.png
/mnt/e/Kitti/training/image_02/0015 000000.png
/mnt/e/Kitti/training/image_02/0016 000000.png
/mnt/e/Kitti/training/image_02/0017 000021.png
/mnt/e/Kitti/training/image_02/0018 000000.png
/mnt/e/Kitti/training/image_02/0019 000545.png
/mnt/e/Kitti/training/image_02/0020 000000.png


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint

import numpy as np
from skimage import data
from skimage.transform import resize

import matplotlib.pyplot as plt

import torch

from src.reader import KineticDataset, KineticDatasetVideo

from src.vqgan import ViTVQGAN
from src.my_model import Encoder, Decoder, MaskVideo
from src.trainer import AttentionMaskModeling


In [ ]:
path = "/mnt/e/kinetics-dataset/k400"
split = "train"

# ds = KineticDatasetVideo(
#     path, split,
#     n_frames=16,
# )
ds = KineticDatasetVideo.get_ds(path, split, 16)
ds_loader = DataLoader(
    ds, 2, True,
    num_workers=4
)

In [ ]:
for batch in ds_loader:
    frames, _ = batch
    break

# _batch = batch[0].cuda(), None
B, T, _, H, W = frames.shape
frames.shape

In [ ]:
is_finetune = False
image_size = (256, 256)
patch_size = (8, 8)
depth, heads, dim, embed_dim = 8, 8, 512, 32

window_size = (3, 3)
length, height, width = 32, 32, 32
temporal_depth, temporal_heads, temporal_dim = 6, 8, 128
n_codes = 8192

vitvq_path = "./checkpoint/imagenet_vitvq_small.ckpt"

In [ ]:
lightning_model = AttentionMaskModeling(
    model=MaskVideo(
        is_finetune=is_finetune,
        
        image_size=image_size, patch_size=patch_size,
        vit_depth=depth, vit_heads=heads, vit_dim=dim,
        n_codes=n_codes, embed_dim=embed_dim,

        window_size=window_size,
        length=length, height=height, width=width,
        temporal_depth=temporal_depth, temporal_heads=temporal_heads, #temporal_dim=temporal_dim,

        drop_prob=0.1, depth_prob=0.1,
        vitvq_path=vitvq_path
	), 
    top_p=0.975,
    lr=1e-4
)
lightning_model.cuda()
lightning_model.model.cuda()
print()

In [ ]:
is_dev = False

trainer = pl.Trainer(
    # training settings
    max_epochs=50,
    val_check_interval=1.0,
    accelerator="cpu" if is_dev else "gpu",
    precision="32-true" if is_dev else "16-mixed",
    accumulate_grad_batches=1,
    gradient_clip_val=1.0,
    # logging settings
    default_root_dir=f"./checkpoints",
    # logger=wandb_logger,
    callbacks=[
        ModelCheckpoint(
			monitor="train_acc",
			dirpath="./trained",
			filename="semcom-{epoch:02d}-{train_acc:.2f}",
			save_top_k=3,
			mode="max",
		)
	],
    # dev setting
    fast_dev_run=is_dev,
)
trainer.fit(
    lightning_model, 
    train_dataloaders=ds_loader,
    # val_dataloaders=None,
)

In [ ]:
frames.min(), frames.max()

In [ ]:
_ = lightning_model.model(frames.cuda())

In [ ]:
lightning_model.model.create_temporal_mask(4, 4)

In [ ]:
mask = lightning_model.model.get_mask_from_frames(frames, 0.95)
mask.shape

In [ ]:
mask[0, 0].sum()

In [ ]:
logits = torch.rand(8).reshape(1, 8)
print("logits", logits)

In [ ]:
attn = logits.softmax(-1)
print("attn  ", attn)

sorted_attn, sorted_indices = torch.topk(attn, attn.shape[-1], largest=True, sorted=True, dim=1)
print("sorted", sorted_attn)
print("sorted", sorted_indices)

cumsum = torch.cumsum(sorted_attn, dim=-1)
print("cumsum", cumsum)

prob = cumsum > 0.4
prob[..., 1:] = prob[..., :-1].clone()
prob[..., 0] = False
print("prob  ", prob)

mask = prob.scatter(
    dim=-1,
    index=sorted_indices,  # while ordered, the indices are of original sequence
    src=prob
).to(torch.long)
print("mask  ", 1-mask)

In [ ]:
1-mask

In [ ]:
window_size = 3
height, width = 3, 3

_padding_size = window_size//2
_size = window_size * window_size
n_heads, head_dim = 2, 4


def get_neighbor(x):
    """
        1/ flatten batch, time, and head dimensions
        2/ 
    """
    BT, _, HW, _ = x.shape

    _x = x.reshape(BT, n_heads, height, width, head_dim)
    _x = _x.flatten(0, 1)  # (BT*n_heads, H, W, D')

    neighbor_x = F.unfold(
        _x.permute(0, 3, 1, 2), window_size,
        padding=_padding_size, stride=1,
    )  # (BT*n_heads, D*9, HW)
    neighbor_x = neighbor_x.reshape(BT, n_heads, head_dim, _size, HW)
    neighbor_x = neighbor_x.permute(0, 4, 1, 3, 2).flatten(0, 1)

    return neighbor_x

B, T = 2, 6
HW = height * width

def make_data():
    arange = torch.arange(HW, dtype=torch.float) + 1
    arange = arange[None, None, :, None].repeat((B*T, n_heads, 1, head_dim))

    for d in range(head_dim):
        arange[:, :, :, d] += 10*(d)
    for n in range(n_heads):
        arange[:, n, :, :] += 100*(n)
    for b in range(B*T):
        arange[b, :, :, :] += 1000*(b)

    return arange

q = make_data()
k =-make_data()
v =-make_data()
_q = q.permute(0, 2, 1, 3).flatten(0, 1).unsqueeze(2)

print(q.shape, k.shape, v.shape, _q.shape)

neighbor_k = get_neighbor(k)
neighbor_v = get_neighbor(v)

_q.shape, neighbor_k.shape, neighbor_v.shape

In [ ]:
b, t, h = 0, 0, 0

print(q[b*t+t, h])
print(_q[b*t*h+t*h+h, h])
print(neighbor_k[b*t*h+t*h+h, h])
print(neighbor_v[b*t*h+t*h+h, h])
